# AI01 — train thử SSL Swin V1 MAE Tiny trên Colab

Quy trình: **git clone → os.chdir → cài thư viện → giải nén → !python -m v1.ai01_train_v1**.
Không nhúng hoặc tạo lại code mô hình trong notebook.

- Bật **Runtime → Change runtime type → GPU** trước khi chạy.
- Upload `Prepared_Dataset.zip` lên Google Drive hoặc Files của Colab; ổ F: trên máy local không dùng trực tiếp được.
- Repo phải có `v1/ai01_train_v1.py`, `utils/ai01_data.py`, `v1/ai01_model_v1.py`, `v1/ai01_transfer_v1.py`, `utils/ai01_requirements.txt`.
- Mặc định chỉ train **3 batch**, validation đủ **72 mẫu**. Đây là smoke test, không phải pretraining đủ chất lượng.
- ZIP chứa 504 NPY `[11,400,400,1]`, uint8, split sẵn **432 train / 72 val** theo well.

## 1. Clone repo và chuyển thư mục làm việc

Nếu đã tự clone đúng thư mục thì cell chỉ `os.chdir()`, không clone lại và không tự git pull.

In [ ]:
import os
import sys
from pathlib import Path

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
REPO_DIR = Path("/content/SwinMAE_FJN")
if not REPO_DIR.exists():
    !git clone https://github.com/hahuy2004/SwinMAE_FJN.git "{REPO_DIR}"
os.chdir(REPO_DIR)
for name in ("v1/ai01_train_v1.py", "utils/ai01_data.py", "v1/ai01_model_v1.py", "v1/ai01_transfer_v1.py", "utils/ai01_requirements.txt"):
    assert Path(name).is_file(), f"Repo thiếu {name}; kiểm tra branch/commit bạn đã push lên GitHub"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Working directory:", Path.cwd())

## 2. Cài môi trường và kiểm tra GPU

Dùng **utils/ai01_requirements.txt**, không dùng requirements.txt RGB gốc ở root repo.
Colab thường có sẵn PyTorch CUDA phù hợp; code yêu cầu torch>=2.6,<3 và Transformers 4.53.1.
Nếu đã import Transformers bản khác trước đó, restart runtime rồi chạy lại.

In [ ]:
%pip install -q -r utils/ai01_requirements.txt

In [ ]:
import numpy as np
import torch
import transformers

print("torch:", torch.__version__, "| transformers:", transformers.__version__)
assert transformers.__version__ == "4.53.1", "Restart runtime nếu vẫn đang dùng Transformers đã import từ trước"
assert torch.cuda.is_available(), "Bật GPU trong Runtime → Change runtime type"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 3. Giải nén dữ liệu ngay trong repo

Sửa `ZIP_PATH` cho đúng file của bạn. Nếu upload bằng Files vào `/content`, đổi
`MOUNT_DRIVE=False` và `ZIP_PATH=Path("/content/Prepared_Dataset.zip")`.

Nếu đã tự giải nén đúng cấu trúc thì cell không làm lại:cl

```text
/content/SwinMAE_FJN/
├── v1/ai01_train_v1.py
├── utils/ai01_data.py
└── Prepared_Dataset/
    ├── train.txt
    ├── validation.txt
    └── Data_Part_sample/unlabelled/<well>/<frame>.npy
```

In [ ]:
MOUNT_DRIVE = True
ZIP_PATH = Path("/content/drive/MyDrive/AI01/Prepared_Dataset.zip")
DATA_ROOT = REPO_DIR / "Prepared_Dataset"

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
import stat
import zipfile
from pathlib import PurePosixPath

if DATA_ROOT.exists():
    assert (DATA_ROOT / "train.txt").is_file() and (DATA_ROOT / "validation.txt").is_file(), "Thư mục data chưa giải nén đủ; kiểm tra trước khi chạy lại"
    print("Dùng data đã giải nén:", DATA_ROOT)
else:
    assert ZIP_PATH.is_file(), f"Không thấy ZIP: {ZIP_PATH}"
    with zipfile.ZipFile(ZIP_PATH) as archive:
        # Chỉ cho ghi bên trong Prepared_Dataset, không ghi đè code repo.
        for info in archive.infolist():
            relative = PurePosixPath(info.filename)
            if (relative.is_absolute() or ".." in relative.parts or "\\" in info.filename
                    or not relative.parts or relative.parts[0] != "Prepared_Dataset"
                    or stat.S_ISLNK(info.external_attr >> 16)):
                raise ValueError(f"ZIP không đúng cấu trúc/an toàn: {info.filename}")
        archive.extractall(REPO_DIR)
    print("Đã giải nén:", DATA_ROOT)

## 4. Kiểm tra data/split

Chỉ kiểm tra, không crop/resize NPY lại. Loader sẽ crop giữa 256×256 và chuẩn hóa
khi train. Không dùng `--data_path Prepared_Dataset` vì sẽ lấy luôn well validation vào train.

In [ ]:
from utils.ai01_data import AI01UnlabelledDataset, ensure_disjoint_wells

train_dataset = AI01UnlabelledDataset(manifest=DATA_ROOT / "train.txt", augment=False)
val_dataset = AI01UnlabelledDataset(manifest=DATA_ROOT / "validation.txt", augment=False)
ensure_disjoint_wells(train_dataset, val_dataset)
assert (len(train_dataset), len(val_dataset)) == (432, 72), "ZIP khác bộ mẫu đã kiểm tra"
referenced = set(train_dataset.files) | set(val_dataset.files)
assert referenced == {p.resolve() for p in DATA_ROOT.rglob("*.npy")}, "Manifest không khớp tập NPY"
for path in sorted(referenced):
    path.relative_to(DATA_ROOT.resolve())
    array = np.load(path, allow_pickle=False, mmap_mode="r")
    assert array.shape == (11, 400, 400, 1) and array.dtype == np.uint8, (path, array.shape, array.dtype)
sample = train_dataset[0]
assert sample.shape == (11, 256, 256) and torch.isfinite(sample).all()
print("Train:", len(train_dataset), "| validation:", len(val_dataset))
print("Input của model:", tuple(sample.shape), sample.dtype)

## 5. Xem thử ảnh và đặt cấu hình train

Mặc định kiểm tra 3 batch. Muốn train lâu hơn, đổi `EPOCHS=20`, `MAX_STEPS=0`,
`WARMUP_EPOCHS=2`, `ACCUM_ITER=4` và chọn **OUTPUT_DIR mới**. Không tự train dài khi Run all.

In [ ]:
import matplotlib.pyplot as plt

array = np.load(train_dataset.files[0], allow_pickle=False)
plt.figure(figsize=(4, 4))
plt.imshow(array[5, 72:328, 72:328, 0], cmap="gray", vmin=0, vmax=255)
plt.title("Z05 — vùng 256x256 mà Swin nhận")
plt.axis("off")
plt.show()

In [ ]:
OUTPUT_DIR = Path("output_ai01_colab_smoke")
EPOCHS = 1
MAX_STEPS = 3       # 0 = dùng toàn bộ batch mỗi epoch
WARMUP_EPOCHS = 0
BATCH_SIZE = 1
ACCUM_ITER = 1

## 6. Train SSL — chạy lệnh trực tiếp

Chỉ cần `!python -m v1.ai01_train_v1 ...`. Script tự tạo Swin V1 Tiny, decoder
tái tạo, optimizer, train/val và checkpoint; notebook không cần dựng lại model.

Kết quả mặc định nằm trong repo: `output_ai01_colab_smoke/`.
Để lưu trực tiếp lên Drive sau mỗi epoch, có thể đặt OUTPUT_DIR thành đường dẫn
Drive tuyệt đối trước khi chạy. Data vẫn đọc từ repo trên ổ local.

In [ ]:
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError("OUTPUT_DIR đã có dữ liệu: chọn tên mới để không ghi đè run cũ")

!python -u -m v1.ai01_train_v1 \
    --manifest Prepared_Dataset/train.txt \
    --val_manifest Prepared_Dataset/validation.txt \
    --output_dir "{OUTPUT_DIR}" \
    --epochs {EPOCHS} \
    --warmup_epochs {WARMUP_EPOCHS} \
    --max_steps {MAX_STEPS} \
    --batch_size {BATCH_SIZE} \
    --accum_iter {ACCUM_ITER} \
    --num_workers 2 \
    --lr 0.0001 \
    --device cuda
assert (OUTPUT_DIR / "encoder-last.pt").is_file(), "Train chưa hoàn tất; xem lỗi ở log phía trên"

## 7. Xem kết quả

- `encoder-best.pt` / `encoder-last.pt`: lấy về nạp vào AI01.
- `checkpoint-last.pt`: full SSL checkpoint để resume, không dùng làm encoder.
- `log.jsonl`: train/validation MSE, không phải độ chính xác segmentation.

Train thử 3 batch chỉ chứng minh pipeline chạy được. Mẫu 7 well cùng session chưa
đủ đánh giá chất lượng pretrain hay khả năng tổng quát hóa.

In [ ]:
import json

history = [json.loads(line) for line in (OUTPUT_DIR / "log.jsonl").read_text().splitlines() if line.strip()]
plt.figure(figsize=(7, 4))
plt.plot([r["epoch"] for r in history], [r["train_loss"] for r in history], "o-", label="train")
plt.plot([r["epoch"] for r in history], [r["validation_loss"] for r in history], "o-", label="validation")
plt.xlabel("Epoch")
plt.ylabel("Masked MSE")
plt.legend()
plt.grid(True)
plt.show()
print(history[-1])
print("Encoder:", (OUTPUT_DIR / "encoder-best.pt").resolve())

## 8. Tùy chọn: kiểm tra chuyển encoder / tải về / backup Drive

Mặc định các thao tác này tắt. Bật biến tương ứng nếu cần.
`python -m v1.ai01_verify_v1` tải **config** Mask2Former từ Hub (không tải pretrained weights),
kiểm tra nạp encoder và forward Mask2Former thường; không chạy decoder BiMamba tùy biến.

**Quan trọng:** repo, data và checkpoint dưới `/content` đều mất khi Colab reset.
Tải encoder về hoặc backup Drive trước khi ngắt runtime. Khi train dài và cần
checkpoint mỗi epoch tồn tại sau reset, đặt OUTPUT_DIR trực tiếp trên Drive.

In [ ]:
VERIFY_TRANSFER = False
DOWNLOAD_ENCODER = False
BACKUP_TO_DRIVE = False
ENCODER_PATH = OUTPUT_DIR / "encoder-best.pt"

if VERIFY_TRANSFER:
    %pip install -q -r utils/ai01_requirements_downstream.txt
    !python -m v1.ai01_verify_v1 --encoder "{ENCODER_PATH}" --device cuda
if DOWNLOAD_ENCODER:
    from google.colab import files
    files.download(str(ENCODER_PATH))
if BACKUP_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    backup = Path("/content/drive/MyDrive/AI01_SSL") / OUTPUT_DIR.name
    assert not backup.exists(), "Đã có backup cùng tên: chọn thư mục khác để không ghi đè"
    shutil.copytree(OUTPUT_DIR, backup)
    print("Backup:", backup)

## Resume và tích hợp AI01 (tham khảo, không tự chạy)

Resume bằng cùng lệnh/đường dẫn/cấu hình cũ, thêm:

```python
!python -m v1.ai01_train_v1 \
    --manifest Prepared_Dataset/train.txt \
    --val_manifest Prepared_Dataset/validation.txt \
    --output_dir output_ai01_colab_smoke \
    --epochs 1 --warmup_epochs 0 --max_steps 3 --batch_size 1 --accum_iter 1 \
    --num_workers 2 --lr 0.0001 --device cuda \
    --resume output_ai01_colab_smoke/checkpoint-last.pt
```

Không đổi số epoch/lịch train khi resume. Nếu smoke đã xong một epoch thì lệnh
trên không train thêm; muốn train dài hãy tạo run mới, không resume smoke với
tham số khác. Sau reset, restore checkpoint từ Drive về đúng đường dẫn và giải
nén cùng dataset trong `/content/SwinMAE_FJN` trước khi resume.

Sau khi tải encoder về máy chạy `A1_Refractor.ipynb`:

```python
SSL_ENCODER_CHECKPOINT = "duong/dan/encoder-best.pt"
SSL_CODE_DIR = Path("duong/dan/AI01/SSL/Swin-MAE")
```

Chạy lại cell khởi tạo model trước Trainer/optimizer. Không tạo lại projection
sau khi nạp SSL. Nếu OOM, giảm batch size; nếu gradient overflow, thử thêm `--no_amp`.